# Cambodia Land-Cover Segmentation — Colab / Kaggle training

Trains the four models (U-Net, SegFormer, standard DeepLabV3+, SWIR-Attention
DeepLabV3+) on a free GPU, each for several seeds, then evaluates on the held-out
test set and prints the mean±std table plus the key SWIR-Attention vs standard
DeepLabV3+ comparison.

**Before running:** set the runtime to GPU (Runtime → Change runtime type → GPU),
and do the one-time data upload described in the *Data* cell below.

The code (and the committed `results/norm_stats.json`) come from GitHub; only the
`data/tiles/` folder is pulled from your Google Drive.

## 0. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Get the code

Clones the repo. If the repo is private, create a GitHub token and use the
commented URL form instead.

In [ ]:
REPO = 'https://github.com/codingwithchitra777/cambodia-landcover-segmentation.git'
BRANCH = 'data-pipeline-scripts'  # change to 'main' after the PR is merged
# Private repo? use: REPO = 'https://<TOKEN>@github.com/codingwithchitra777/cambodia-landcover-segmentation.git'

%cd /content
!rm -rf cambodia-landcover-segmentation
!git clone -b $BRANCH $REPO
%cd cambodia-landcover-segmentation

## 2. Install dependencies

Colab already has torch. We only need the segmentation libraries for the models
(training does not need rasterio/geopandas/earthengine — those were for building
the tiles).

In [ ]:
!pip -q install segmentation-models-pytorch transformers pyyaml

## 3. Data — one-time upload, then unzip from Drive

The tiles (`data/tiles/`) are not in git. Do this **once**, on your own machine:

```bash
# from the repo root, zip the tiles + manifests
cd data && zip -r tiles.zip tiles && cd ..
```
then upload `data/tiles.zip` to your Google Drive (anywhere; note the path).

The cell below mounts Drive and unzips it into `data/tiles/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path to the tiles.zip you uploaded to Drive:
TILES_ZIP = '/content/drive/MyDrive/tiles.zip'

!mkdir -p data
!unzip -oq "$TILES_ZIP" -d data/
!ls data/tiles | head
import glob; print('tiles:', len(glob.glob('data/tiles/*.npz')))

## 4. Normalization stats

`results/norm_stats.json` is committed in the repo (computed from the train
split). If your tiles changed, recompute it; otherwise this is a no-op.

In [ ]:
import os
if not os.path.exists('results/norm_stats.json'):
    !python src/dataset.py --manifest data/tiles/manifest_train.csv --tiles-dir data/tiles --out results/norm_stats.json
else:
    print('using committed results/norm_stats.json')

## 5. Train all four models × seeds

Baselines and standard DeepLabV3+ use cross-entropy; SWIR-Attention uses the
boundary-aware loss (its modification 3). All use class weights + built-up
oversampling + augmentation. Adjust `SEEDS` / `EPOCHS` as time allows (>=3 seeds
for the thesis).

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 50
BATCH = 8

# (model, loss)
RUNS = [
    ('unet',           'ce'),
    ('segformer',      'ce'),
    ('deeplabv3plus',  'ce'),             # the baseline to beat
    ('swir_attention', 'boundary_aware'), # the contribution
]

for model, loss in RUNS:
    for seed in SEEDS:
        print(f'\n===== {model} ({loss}) seed {seed} =====')
        !python src/train.py --model $model --loss $loss --seed $seed \
            --epochs $EPOCHS --batch-size $BATCH --oversample --num-workers 2

## 6. Evaluate every checkpoint on the TEST set

In [ ]:
import glob
for ckpt in sorted(glob.glob('models/*.pth')):
    !python src/evaluate.py --checkpoint "$ckpt" --num-workers 2

## 7. The thesis table + key comparison

In [ ]:
!python src/evaluate.py --aggregate results

## 8. Save results + checkpoints back to Drive

Colab is ephemeral — copy everything you want to keep to Drive before the runtime
recycles.

In [ ]:
!mkdir -p /content/drive/MyDrive/cambodia_lc_results
!cp -r results /content/drive/MyDrive/cambodia_lc_results/
!cp -r models  /content/drive/MyDrive/cambodia_lc_results/
print('copied results/ and models/ to Drive/cambodia_lc_results')